# The pieces of a GPT, computed by hand

A GPT is a short list of parts: scaled dot-product attention, two masks, several heads, a stack of pre-norm blocks, a subword tokenizer, a next-token loss and the Adam optimizer with a learning-rate schedule. This notebook does not train a model end to end. Instead, it takes each part on its own, computes it on numbers small enough to check by hand, and reproduces every worked number from the slides.

| Part | What you build | Numbers from the slides you reproduce |
|---|---|---|
| 1 | Softmax with a temperature; why we divide by $\sqrt{D}$ | $[0.9997, 0.0003, 0.0000]$ vs $[0.665, 0.245, 0.090]$; the $\tau = 0.5, 1, 2$ table |
| 2 | The padding mask, the causal mask, and their AND | 15 entries of a $6\times6$ causal mask; the shapes for $B=32, T=42, D=64$ |
| 3 | Multi-head attention, checked against PyTorch; word order; the KV cache | the 2.1 GB cache for a 32-layer, $D=4096$ model |
| 4 | Parameter counts for one layer and for a full GPT | 11,620 for $V=100, T=64, D=32$; GPT-2 small and XL; GPT-3 |
| 5 | Byte-pair encoding on a toy corpus | `l o w e r → lo w e r → low e r → low er` |
| 6 | The shifted next-token loss and perplexity | perplexity 1 is perfect, $V$ is a uniform guess |
| 7 | Adam, AdamW and warmup + cosine | the first Adam step (both weights move 0.001); SGD 68 → 1.45 vs Adam 68 → 0.003; half the peak at step 5,250 |

**How it works.** Each part has a short explanation, then a `TODO` cell with a small function to write, then a check cell. Until you fill a `TODO`, its check cell prints a reminder and skips, so the notebook always runs top to bottom. Everything runs on a CPU in well under a minute, with fixed seeds and no downloads.

**Notation (as on the slides).** Row form, as in PyTorch: $X$ is $T \times D$ (tokens by features), $Q = XW_q$, $K = XW_k$, $V = XW_v$, scores $S = QK^\top/\sqrt{D}$ with $S[t, i] = e_{i,t}/\sqrt{D}$, weights $A = \mathrm{softmax}(S)$ along each row with $A[t, i] = \alpha_{i,t}$, output $Z = AV$ whose row $t$ is $c_t$. Query $t$ asks, key $i$ is looked at. $B$ is the batch size. In parameter counts $V$ is the vocabulary size (the slides use the letter for both; the context tells you which). Adam: learning rate $\gamma$, gradient $g_t$, running averages $m_t$ and $s_t$, decay rates $\beta_1, \beta_2$, bias-corrected $\hat m_t, \hat s_t$, and $\epsilon$. Temperature $\tau$, logits $z$.

In [1]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(2)
torch.manual_seed(0)
np.random.seed(0)
T_START = time.time()

def attempt(fn, *args, **kwargs):
    """Run a TODO function; if it is not written yet, say so and return None instead of crashing."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"[TODO] {fn.__name__} is not implemented yet: skipping this check.")
        return None

print("torch", torch.__version__)

torch 2.9.1


## Part 1. Softmax, temperature, and why we divide by $\sqrt{D}$

Attention turns each row of scores into weights with a softmax, and decoding turns the last position's logits $z$ into next-token probabilities the same way, after dividing by a temperature $\tau$:

$$p_j = \frac{e^{z_j/\tau}}{\sum_k e^{z_k/\tau}}.$$

Two practical points. Subtract the maximum before exponentiating (it cancels, and it stops $e^{z}$ from overflowing). And an entry of $-\infty$ must come out as exactly 0: that is how masks work in Part 2.

**The scaling argument.** If $q$ and $k$ have $D$ independent entries with mean 0 and variance 1, then $q\cdot k$ is a sum of $D$ such products, so its variance is $D$. With $D = 64$ the scores have standard deviation about 8, the softmax becomes nearly one-hot, and its gradient nearly vanishes. Dividing by $\sqrt{D} = 8$ brings the variance back to 1.

In [2]:
# TODO 1: a numerically stable softmax with a temperature.
def softmax_t(z, tau=1.0):
    """z: 1-D numpy array of logits (entries may be -inf). Return softmax(z / tau) as a numpy array.
    Hint: divide by tau, subtract the max, exponentiate, normalise."""
    raise NotImplementedError

In [3]:
p_raw = attempt(softmax_t, np.array([12.0, 4.0, -4.0]))
if p_raw is not None:
    p_scaled = softmax_t(np.array([12.0, 4.0, -4.0]) / 8)
    print("raw    q.k     [12, 4, -4]     ->", [f"{v:.4f}" for v in p_raw])
    print("scaled q.k / 8 [1.5, 0.5, -0.5] ->", [f"{v:.3f}" for v in p_scaled])
    assert np.allclose(p_raw, [0.9997, 0.0003, 0.0000], atol=5e-5)
    assert np.allclose(p_scaled, [0.665, 0.245, 0.090], atol=5e-4)

    # the temperature table, z = [3.0, 1.5, 0.5, -1.0]
    z = np.array([3.0, 1.5, 0.5, -1.0])
    table = {0.5: [0.95, 0.05, 0.01, 0.00], 1.0: [0.76, 0.17, 0.06, 0.01], 2.0: [0.53, 0.25, 0.15, 0.07]}
    print("\ntau    a     o     e     space")
    for tau, row in table.items():
        p = softmax_t(z, tau)
        print(f"{tau:3.1f}  " + "  ".join(f"{x:.2f}" for x in p))
        assert np.allclose(p.round(2), row), tau

    # stability and masking
    assert np.allclose(softmax_t(np.array([1000.0, 1000.0])), [0.5, 0.5])
    assert softmax_t(np.array([0.0, -np.inf, 1.0]))[1] == 0.0
    print("stable for large logits, exactly 0 for -inf: ok")

[TODO] softmax_t is not implemented yet: skipping this check.


Now the variance claim itself, by simulation. We also measure how much signal can flow back through the softmax: the size of its Jacobian $\partial p/\partial z = \mathrm{diag}(p) - pp^\top$.

In [4]:
g = torch.Generator().manual_seed(0)
D = 64
q = torch.randn(20000, D, generator=g); k = torch.randn(20000, D, generator=g)
dots = (q * k).sum(1)
print(f"D = {D}: var(q.k) = {dots.var():.1f}  (theory {D}),  var(q.k / sqrt(D)) = {(dots / math.sqrt(D)).var():.2f}  (theory 1)")
assert abs(dots.var().item() / D - 1) < 0.05

def softmax_jacobian_norm(z):
    p = torch.softmax(torch.as_tensor(z, dtype=torch.float64), 0)
    return torch.linalg.matrix_norm(torch.diag(p) - torch.outer(p, p)).item()

j_raw, j_scaled = softmax_jacobian_norm([12.0, 4.0, -4.0]), softmax_jacobian_norm([1.5, 0.5, -0.5])
print(f"softmax Jacobian norm: raw scores {j_raw:.5f}, scaled scores {j_scaled:.3f}  ({j_scaled / j_raw:.0f}x larger)")
assert j_scaled > 100 * j_raw

D = 64: var(q.k) = 63.2  (theory 64),  var(q.k / sqrt(D)) = 0.99  (theory 1)
softmax Jacobian norm: raw scores 0.00067, scaled scores 0.389  (581x larger)


With raw scores almost no gradient gets through the softmax, so the model cannot learn which keys to prefer. That is the whole reason for the $1/\sqrt{D}$, and why the temperature table looks the way it does: dividing logits by a number larger than 1 flattens the distribution.

## Part 2. Masks: padding, causal, and both

Masks are boolean matrices `allowed[b, t, i]`: may query $t$ look at key $i$ in sequence $b$? Disallowed scores are set to $-\infty$ **before** the softmax.

* **Key-padding mask.** Sequences in a batch are right-padded to the same length. No query may look at a `<pad>` key: given a 0/1 `pad_mask` of shape $(B, T)$ (1 = real token), column $i$ is allowed only if `pad_mask[b, i] == 1`.
* **Causal mask.** Query $t$ may only look at keys $i \le t$: a lower-triangular matrix, `torch.tril`.
* **Both.** Combine with a logical AND, broadcast over the batch.

The slide's attention function is below; you write the mask it needs.

In [5]:
def masked_attention(Q, K, V, allowed):
    # Q, K, V: (B, T, D)     allowed: (B, T, T) bool
    D = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(D)    # (B, T, T)
    S = S.masked_fill(~allowed, float('-inf'))   # mask BEFORE softmax
    A = torch.softmax(S, dim=-1)                 # each row sums to 1
    return A @ V, A                              # (B, T, D), weights

In [6]:
# TODO 2: the combined causal AND key-padding mask.
def build_allowed(T, pad_mask=None):
    """T: sequence length. pad_mask: (B, T) tensor of 0/1 (1 = real token), or None for no padding.
    Return a bool tensor `allowed` of shape (B, T, T) (B = 1 if pad_mask is None) with
    allowed[b, t, i] = (i <= t) and pad_mask[b, i] == 1.
    Hint: torch.tril(torch.ones(T, T, dtype=torch.bool)), then & pad_mask.bool()[:, None, :]."""
    raise NotImplementedError

In [7]:
allowed6 = attempt(build_allowed, 6)
if allowed6 is not None:
    n_masked = (~allowed6[0]).sum().item()
    print("6 x 6 causal mask (1 = allowed):\n", allowed6[0].int().numpy())
    print("entries set to -inf:", n_masked, "= 6 * 5 / 2")
    assert allowed6.shape == (1, 6, 6) and n_masked == 15

    # a length-6 sequence whose last two positions are <pad>
    pad = torch.tensor([[1, 1, 1, 1, 0, 0]])
    both = build_allowed(6, pad)[0]
    extra = (~both & allowed6[0]).nonzero().tolist()
    print("extra entries masked by padding (query t, key i):", extra)
    assert all(i in (4, 5) for _, i in extra) and len(extra) == 3   # (4,4), (5,4), (5,5): columns 4 and 5

    # shapes for a batch of 32 sequences, T = 42, D = 64, and a check against PyTorch's own attention
    B, T, D = 32, 42, 64
    g = torch.Generator().manual_seed(1)
    Q, K, V = (torch.randn(B, T, D, generator=g) for _ in range(3))
    lengths = torch.randint(20, T + 1, (B,), generator=g)
    pad_mask = (torch.arange(T)[None, :] < lengths[:, None]).long()
    allowed = build_allowed(T, pad_mask)
    Z, A = masked_attention(Q, K, V, allowed)
    print(f"\nQ {tuple(Q.shape)}, K^T {tuple(K.transpose(-2, -1).shape)}, S {tuple(A.shape)}, Z {tuple(Z.shape)}")
    assert A.shape == (32, 42, 42) and Z.shape == (32, 42, 64)
    assert torch.allclose(A.sum(-1), torch.ones(B, T), atol=1e-6)
    assert (A[~allowed] == 0).all()
    ref = F.scaled_dot_product_attention(Q, K, V, attn_mask=allowed)
    assert torch.allclose(Z, ref, atol=1e-5)
    print("rows sum to 1, masked weights are exactly 0, matches F.scaled_dot_product_attention: ok")

[TODO] build_allowed is not implemented yet: skipping this check.


**Two checks from the slides.** First, *mask after the softmax* is wrong: zeroing the weights afterwards leaves rows that no longer sum to 1, and the masked scores already shaped the denominator. Second, the *no-information-from-the-future* test: change every token after position $t$ and confirm that the outputs at positions $0, \dots, t$ do not move.

In [8]:
if allowed6 is not None:
    g = torch.Generator().manual_seed(2)
    Q, K, V = (torch.randn(1, 6, 8, generator=g) for _ in range(3))
    S = Q @ K.transpose(-2, -1) / math.sqrt(8)
    A_wrong = torch.softmax(S, -1) * allowed6          # mask AFTER the softmax
    print("row sums when masking after the softmax:", A_wrong.sum(-1)[0].numpy().round(3))
    assert not torch.allclose(A_wrong.sum(-1), torch.ones(1, 6))

    # future-leak test on a random input
    t = 3
    X = torch.randn(1, 6, 8, generator=g)
    X2 = X.clone(); X2[:, t + 1:] = torch.randn(1, 6 - t - 1, 8, generator=g)    # new future
    Wq, Wk, Wv = (torch.randn(8, 8, generator=g) for _ in range(3))
    y1 = masked_attention(X @ Wq, X @ Wk, X @ Wv, allowed6)[0]
    y2 = masked_attention(X2 @ Wq, X2 @ Wk, X2 @ Wv, allowed6)[0]
    print("outputs at positions 0..t unchanged:", torch.allclose(y1[:, :t + 1], y2[:, :t + 1], atol=1e-6))
    assert torch.allclose(y1[:, :t + 1], y2[:, :t + 1], atol=1e-6)
    no_mask = torch.ones(1, 6, 6, dtype=torch.bool)
    z1 = masked_attention(X @ Wq, X @ Wk, X @ Wv, no_mask)[0]
    z2 = masked_attention(X2 @ Wq, X2 @ Wk, X2 @ Wv, no_mask)[0]
    print("without the causal mask, the same test fails:", not torch.allclose(z1[:, :t + 1], z2[:, :t + 1], atol=1e-6))

## Part 3. Multi-head attention, word order, and the KV cache

With $H$ heads, each head $h$ has its own queries, keys and values of width $d = D/H$. In practice we keep the three $D \times D$ matrices $W_q, W_k, W_v$ and **split** their outputs into $H$ slices of width $d$. Each head runs scaled dot-product attention (scaled by $\sqrt{d}$, the head width), the $H$ outputs $c_t^{(1)}, \dots, c_t^{(H)}$ are concatenated back to width $D$, and one more matrix $W_O$ ($D \times D$) mixes them.

Shapes, for an input $X$ of shape $(B, T, D)$:

$$XW_q:\ (B, T, D) \;\to\; \text{view } (B, T, H, d) \;\to\; \text{transpose } (B, H, T, d) \;\to\; \text{scores } (B, H, T, T) \;\to\; \text{outputs } (B, H, T, d) \;\to\; (B, T, D).$$

In [9]:
# TODO 3: multi-head self-attention in row form (no biases).
def multi_head_attention(X, Wq, Wk, Wv, Wo, H, allowed=None):
    """X: (B, T, D). Wq, Wk, Wv, Wo: (D, D), used as X @ W. H: number of heads (D divisible by H).
    allowed: None (no mask) or a bool tensor broadcastable to (B, H, T, T), e.g. (B, 1, T, T).
    Return Z of shape (B, T, D) = concat_heads(attention per head) @ Wo.
    Steps: project; split into heads (B, H, T, d); scores / sqrt(d); mask with -inf; softmax over keys;
    weight the values; merge heads back to (B, T, D); multiply by Wo."""
    raise NotImplementedError

In [10]:
torch.manual_seed(3)
B, T, D, H = 2, 7, 16, 4
mha_ref = nn.MultiheadAttention(D, H, bias=False, batch_first=True)
Wq, Wk, Wv = (w.T.detach() for w in mha_ref.in_proj_weight.chunk(3))   # PyTorch stores W^T
Wo = mha_ref.out_proj.weight.T.detach()
X = torch.randn(B, T, D)
Z = attempt(multi_head_attention, X, Wq, Wk, Wv, Wo, H)
if Z is not None:
    with torch.no_grad():
        Z_ref = mha_ref(X, X, X, need_weights=False)[0]
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
        Zc_ref = mha_ref(X, X, X, attn_mask=~causal, need_weights=False)[0]   # PyTorch: True = blocked
    Zc = multi_head_attention(X, Wq, Wk, Wv, Wo, H, causal[None, None])
    print("output shape:", tuple(Z.shape))
    print("matches nn.MultiheadAttention, no mask:", torch.allclose(Z, Z_ref, atol=1e-5))
    print("matches nn.MultiheadAttention, causal: ", torch.allclose(Zc, Zc_ref, atol=1e-5))
    assert Z.shape == (B, T, D) and torch.allclose(Z, Z_ref, atol=1e-5) and torch.allclose(Zc, Zc_ref, atol=1e-5)
    # with one head and W_O = I it is the single-head attention of Part 2
    Z1 = multi_head_attention(X, Wq, Wk, Wv, torch.eye(D), 1, causal[None, None])
    assert torch.allclose(Z1, masked_attention(X @ Wq, X @ Wk, X @ Wv, causal[None].expand(B, T, T))[0], atol=1e-5)
    print("H = 1 and W_O = I gives the single-head masked attention: ok")

[TODO] multi_head_attention is not implemented yet: skipping this check.


### Does self-attention know word order?

The slides ask: shuffle the rows of $X$ with a permutation matrix $P$. What happens to $\mathrm{softmax}(QK^\top/\sqrt{D})V$, and to a mean pool over tokens? Which part of a GPT makes order visible? Here is the experiment, using your function.

In [11]:
if Z is not None:
    g = torch.Generator().manual_seed(4)
    perm = torch.randperm(T, generator=g)
    Xp = X[:, perm]                                         # "dog bites man" -> "man bites dog"
    f = lambda inp, mask=None: multi_head_attention(inp, Wq, Wk, Wv, Wo, H, mask)
    eq = torch.allclose(f(Xp), f(X)[:, perm], atol=1e-5)
    pool = torch.allclose(f(Xp).mean(1), f(X).mean(1), atol=1e-5)
    print("no positions, no mask:   outputs are permuted the same way:", eq, "| mean pool unchanged:", pool)
    pos = torch.randn(1, T, D, generator=g)                 # a learned position embedding, one vector per position
    pool_pos = torch.allclose(f(Xp + pos).mean(1), f(X + pos).mean(1), atol=1e-5)
    print("with position embeddings: mean pool unchanged:", pool_pos)
    cm = causal[None, None]
    pool_causal = torch.allclose(f(Xp, cm).mean(1), f(X, cm).mean(1), atol=1e-5)
    print("with the causal mask only: mean pool unchanged:", pool_causal)
    assert eq and pool and not pool_pos and not pool_causal

Without positions, self-attention is *permutation-equivariant*: shuffle the tokens and the outputs shuffle the same way, so a mean-pooled classifier cannot tell "dog bites man" from "man bites dog". Adding a position vector $p_i$ to each input breaks that symmetry; that is what `pos_embed` is for. Notice that the causal mask also breaks it: position $t$ sees exactly $t+1$ tokens, so a decoder gets some order information even without position embeddings.

### The KV cache

Under the causal mask an old token never attends to a new one, so its key and value never change. When generating, compute $q, k, v$ for the new token only, append $k, v$ to the cache, and attend over the cache. The check below confirms that this gives exactly the last row of full causal attention, then computes the slide's memory cost.

In [12]:
g = torch.Generator().manual_seed(5)
Dk, Tk = 8, 10
Wq1, Wk1, Wv1 = (torch.randn(Dk, Dk, generator=g) for _ in range(3))
Xs = torch.randn(1, Tk, Dk, generator=g)
full = masked_attention(Xs @ Wq1, Xs @ Wk1, Xs @ Wv1, torch.tril(torch.ones(Tk, Tk, dtype=torch.bool))[None])[0]
K_cache, V_cache, rows = [], [], []
for t in range(Tk):                       # one new token per step
    x_new = Xs[:, t]
    K_cache.append(x_new @ Wk1); V_cache.append(x_new @ Wv1)
    Kc, Vc = torch.stack(K_cache, 1), torch.stack(V_cache, 1)       # (1, t+1, D)
    a = torch.softmax((x_new @ Wq1)[:, None] @ Kc.transpose(-2, -1) / math.sqrt(Dk), -1)
    rows.append((a @ Vc)[:, 0])
assert torch.allclose(torch.stack(rows, 1), full, atol=1e-5)
print("cached step-by-step outputs equal full causal attention: ok")

def kv_cache_bytes(layers, D, T, bytes_per_number=2):
    return 2 * layers * T * D * bytes_per_number          # a key and a value, per layer, per token
b = kv_cache_bytes(32, 4096, 4096)
print(f"32 layers, D = 4096, 16-bit, 4,096 tokens: {b:,} bytes = {b / 1e9:.1f} GB per sequence")
assert round(b / 1e9, 1) == 2.1

cached step-by-step outputs equal full causal attention: ok
32 layers, D = 4096, 16-bit, 4,096 tokens: 2,147,483,648 bytes = 2.1 GB per sequence


## Part 4. Counting parameters

The slides count one layer with one head:

| Component | Shape | Count |
|---|---|---|
| `tok_embed` | $V \times D$ | $VD$ |
| `pos_embed` | $T \times D$ | $TD$ |
| $W_q, W_k, W_v$ (no bias) | $3 \times (D\times D)$ | $3D^2$ |
| head (+ bias) | $D \times V + V$ | $DV + V$ |

A full GPT block (GPT-2 style, **pre-LN**: the LayerNorm comes *before* each sub-layer, inside the residual branch) is

$$x \leftarrow x + \mathrm{Attn}(\mathrm{LN}_1(x)), \qquad x \leftarrow x + \mathrm{MLP}(\mathrm{LN}_2(x)),$$

with biases everywhere: LN$_1$ and LN$_2$ have a scale and a shift of size $D$ each; the attention has $W_q, W_k, W_v$ with biases and $W_O$ with a bias; the MLP goes $D \to 4D \to D$ with biases. After the last block there is one more LayerNorm, and the output head shares (ties) its weights with `tok_embed`, so it adds nothing.

Write both counts. The check builds the actual PyTorch modules and compares with `sum(p.numel() for p in model.parameters())`.

In [13]:
# TODO 4: parameter counts.
def one_layer_params(V, T, D):
    """The slide's one-layer, one-head model: tok_embed + pos_embed + W_q, W_k, W_v (no bias) + head with bias."""
    raise NotImplementedError

def gpt_params(V, T, D, L):
    """A GPT-2 style model: tok_embed (V x D) + pos_embed (T x D) + L pre-LN blocks + a final LayerNorm,
    with the output head tied to tok_embed. Each block: 2 LayerNorms, W_q/W_k/W_v/W_O with biases,
    and an MLP D -> 4D -> D with biases. Return the exact total."""
    raise NotImplementedError

In [14]:
class OneLayer(nn.Module):
    def __init__(self, V, T, D):
        super().__init__()
        self.tok_embed, self.pos_embed = nn.Embedding(V, D), nn.Embedding(T, D)
        self.W_q, self.W_k, self.W_v = (nn.Linear(D, D, bias=False) for _ in range(3))
        self.head = nn.Linear(D, V)

class Block(nn.Module):                        # pre-LN: x + Attn(LN(x)), then x + MLP(LN(x))
    def __init__(self, D, H):
        super().__init__()
        self.H = H
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.qkv, self.proj = nn.Linear(D, 3 * D), nn.Linear(D, D)
        self.mlp = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = (t.view(B, T, self.H, D // self.H).transpose(1, 2) for t in self.qkv(self.ln1(x)).chunk(3, -1))
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(a.transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, V, T, D, L, H):
        super().__init__()
        self.tok_embed, self.pos_embed = nn.Embedding(V, D), nn.Embedding(T, D)
        self.blocks = nn.ModuleList(Block(D, H) for _ in range(L))
        self.ln_f = nn.LayerNorm(D)
        self.head = nn.Linear(D, V, bias=False)
        self.head.weight = self.tok_embed.weight          # weight tying
    def forward(self, idx):
        x = self.tok_embed(idx) + self.pos_embed(torch.arange(idx.shape[1]))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))

numel = lambda m: sum(p.numel() for p in m.parameters())    # parameters() lists a tied tensor once

n1 = attempt(one_layer_params, 100, 64, 32)
if n1 is not None:
    print(f"one layer, V=100, T=64, D=32: formula {n1:,}, PyTorch {numel(OneLayer(100, 64, 32)):,}")
    assert n1 == 11_620 == numel(OneLayer(100, 64, 32))
ng = attempt(gpt_params, 65, 32, 48, 3)
if ng is not None:
    torch.manual_seed(6)
    tiny = TinyGPT(65, 32, 48, 3, 4)
    print(f"tiny GPT, V=65, T=32, D=48, L=3: formula {ng:,}, PyTorch {numel(tiny):,}")
    assert ng == numel(tiny)
    # the pre-LN block is still causal: change the future, the past logits stay put
    idx = torch.randint(0, 65, (1, 20)); idx2 = idx.clone(); idx2[:, 11:] = torch.randint(0, 65, (1, 9))
    with torch.no_grad():
        assert torch.allclose(tiny(idx)[:, :11], tiny(idx2)[:, :11], atol=1e-5)
    print("tiny GPT passes the future-leak test: ok")

[TODO] one_layer_params is not implemented yet: skipping this check.
[TODO] gpt_params is not implemented yet: skipping this check.


Now scale the same formula up. GPT-2 uses $V = 50{,}257$ and a context of $T = 1024$; its smallest model has $L = 12, D = 768$ and its largest $L = 48, D = 1600$. GPT-3 175B has $L = 96$, $D = 12{,}288$ and $T = 2048$. Most of the parameters sit in the $12D^2$ per block: 4 $D^2$ in attention and 8 $D^2$ in the MLP.

In [15]:
if ng is not None:
    rows = [("GPT-2 small", 50257, 1024, 768, 12), ("GPT-2 XL", 50257, 1024, 1600, 48),
            ("GPT-3 175B", 50257, 2048, 12288, 96)]
    print(f"{'model':12s} {'L':>3s} {'D':>6s} {'total':>17s} {'12 D^2 L share':>15s}")
    for name, V_, T_, D_, L_ in rows:
        n = gpt_params(V_, T_, D_, L_)
        print(f"{name:12s} {L_:3d} {D_:6d} {n:17,d} {12 * D_**2 * L_ / n:15.1%}")
    assert gpt_params(50257, 1024, 768, 12) == 124_439_808
    assert gpt_params(50257, 1024, 1600, 48) == 1_557_611_200
    assert abs(gpt_params(50257, 2048, 12288, 96) / 1e9 - 175) < 1

Two things to notice. The block term $12D^2L$ is almost the whole count once $D$ is large: for GPT-3 it is over 99%. And the GPT-2 paper's table lists the smallest model as 117M parameters, but counting the released weights gives 124M (the paper's figure is commonly attributed to a counting slip); the largest, listed as 1542M, counts to about 1.56B. A formula you can check against `numel` is more reliable than a number in a table.

## Part 5. Byte-pair encoding

BPE builds a subword vocabulary from a corpus: start from characters, then **repeatedly merge the most frequent adjacent pair** of symbols into one new symbol. Each merge is recorded, and to tokenize new text you replay the merges in the same order.

We use a toy corpus of word counts. Each word starts as a tuple of characters. One merge step:

1. Count every adjacent pair, weighting by the word's count.
2. Pick the most frequent pair (break ties by the smaller pair in alphabetical order, so results are reproducible).
3. Replace every occurrence of that pair, in every word, by the joined symbol.

In [16]:
CORPUS = {"low": 5, "lower": 3, "lot": 2, "her": 2, "newer": 2}
print("corpus:", CORPUS)

corpus: {'low': 5, 'lower': 3, 'lot': 2, 'her': 2, 'newer': 2}


In [17]:
# TODO 5: learn BPE merges, then encode a word with them.
def learn_bpe(corpus, n_merges):
    """corpus: dict word -> count. Return the list of merges, each a pair of symbols like ('l', 'o').
    Start from words split into characters, e.g. ('l', 'o', 'w'). At each step count adjacent pairs
    (weighted by word counts), take the most frequent (ties: the smallest pair), and merge it everywhere."""
    raise NotImplementedError

def bpe_encode(word, merges):
    """Split word into characters, then apply the merges in order. Return the list of tokens."""
    raise NotImplementedError

In [18]:
merges = attempt(learn_bpe, CORPUS, 3)
if merges is not None:
    print("first three merges:", merges)
    assert merges == [("l", "o"), ("lo", "w"), ("e", "r")]
    trace = [" ".join(bpe_encode("lower", merges[:k])) for k in range(4)]
    print("  ->  ".join(trace))
    assert trace == ["l o w e r", "lo w e r", "low e r", "low er"]

    more = learn_bpe(CORPUS, 10)
    print("\nten merges:", more)
    for w in ["lower", "lowest", "newer", "hello"]:
        print(f"  {w:7s} -> {bpe_encode(w, more)}")
    n_char = sum(len(w) * c for w, c in CORPUS.items())
    n_bpe = sum(len(bpe_encode(w, more)) * c for w, c in CORPUS.items())
    print(f"corpus length: {n_char} characters -> {n_bpe} BPE tokens")
    assert all("".join(bpe_encode(w, more)) == w for w in ["lowest", "hello"]) and n_bpe < n_char / 2

[TODO] learn_bpe is not implemented yet: skipping this check.


The trace is the slide's `l o w e r → lo w e r → low e r → low er`. Frequent words end up as one token, words never seen (`lowest`, `hello`) still split into known pieces, and in the worst case into single characters (real tokenizers fall back to bytes), so nothing is ever unknown. The corpus gets shorter as tokens grow: the middle ground between characters and words in the slide's table.

## Part 6. The next-token loss and perplexity

Targets are the inputs shifted left by one. For a batch of token ids `x` of shape $(B, T)$: the model reads `inp = x[:, :-1]`, predicts logits of shape $(B, T{-}1, V)$, and is scored against `tgt = x[:, 1:]` with cross-entropy averaged over all real positions. `<pad>` (id 0) targets are ignored. **Perplexity** is $e^{\text{average loss}}$: 1 is perfect, and $V$ is exactly what a uniform guess over $V$ tokens gets.

We use the slide's example, `"the cat."`, with a character vocabulary.

In [19]:
chars = sorted(set("the cat."))
stoi = {c: i + 1 for i, c in enumerate(chars)}       # 0 is <pad>
VOC = len(chars) + 1
encode = lambda s: [stoi[c] for c in s]
x = torch.tensor([encode("the cat."), encode("the c") + [0, 0, 0]])   # the second sequence is right-padded
print("vocabulary size V =", VOC, " (", chars, "+ <pad> )")
for a, b in zip("the cat.", "the cat."[1:]):
    print(f"  input {a!r} -> target {b!r}")

vocabulary size V = 8  ( [' ', '.', 'a', 'c', 'e', 'h', 't'] + <pad> )
  input 't' -> target 'h'
  input 'h' -> target 'e'
  input 'e' -> target ' '
  input ' ' -> target 'c'
  input 'c' -> target 'a'
  input 'a' -> target 't'
  input 't' -> target '.'


In [20]:
# TODO 6: the shifted next-token loss, and perplexity.
def next_token_loss(model, x, pad_id=0):
    """model: maps token ids (B, T-1) to logits (B, T-1, V). x: (B, T) token ids.
    Shift by one, run the model on the inputs, and return the mean cross-entropy over the
    targets that are not pad_id (a scalar tensor). Hint: reshape logits to (-1, V) and targets to (-1)."""
    raise NotImplementedError

def perplexity(loss):
    """e to the average loss, as a float."""
    raise NotImplementedError

In [21]:
uniform = lambda inp: torch.zeros(*inp.shape, VOC)               # the same score for every token
L_u = attempt(next_token_loss, uniform, x)
if L_u is not None:
    print(f"uniform model: loss {L_u:.4f} = ln V = {math.log(VOC):.4f}, perplexity {perplexity(L_u):.2f} = V")
    assert abs(perplexity(L_u) - VOC) < 1e-4

    # a bigram model that has counted "the cat.": after 't' it has seen 'h' once and '.' once
    counts = torch.zeros(VOC, VOC)
    for a, b in zip(encode("the cat."), encode("the cat.")[1:]):
        counts[a, b] += 1
    probs = counts / counts.sum(1, keepdim=True).clamp(min=1)
    bigram = lambda inp: torch.log(probs[inp] + 1e-12)
    L_b = next_token_loss(bigram, x[:1])
    print(f"bigram model on 'the cat.': loss {L_b:.4f}, perplexity {perplexity(L_b):.4f}  (= 2^(2/7) = {2 ** (2 / 7):.4f})")
    assert abs(perplexity(L_b) - 2 ** (2 / 7)) < 1e-4
    oracle = lambda inp: 50.0 * F.one_hot(x[:1, 1:], VOC).float()      # cheats: it is handed the targets
    L_o = next_token_loss(oracle, x[:1])
    print(f"a model that is handed the answers: perplexity {perplexity(L_o):.4f}")
    assert perplexity(L_o) < 1.0001

    # pads are ignored: the padded row contributes exactly its 4 real targets
    torch.manual_seed(7)
    rnd = nn.Sequential(nn.Embedding(VOC, 8), nn.Linear(8, VOC))
    logp = torch.log_softmax(rnd(x[:, :-1]), -1)
    tgt = x[:, 1:]
    manual = -(logp.gather(-1, tgt[..., None])[..., 0] * (tgt != 0)).sum() / (tgt != 0).sum()
    assert torch.allclose(next_token_loss(rnd, x), manual, atol=1e-6) and (tgt != 0).sum() == 7 + 4
    print("matches a hand-written masked average over the 11 real targets: ok")

[TODO] next_token_loss is not implemented yet: skipping this check.


The bigram model is perfect everywhere except after `t`, where it has seen `h` once and `.` once and so gives each probability $\tfrac12$. Two of the seven predictions cost $\ln 2$ each, so the average loss is $\tfrac{2}{7}\ln 2$ and the perplexity is $2^{2/7} \approx 1.22$: "on average as uncertain as a choice between 1.22 options". One sentence of $T$ tokens gives $T-1$ training examples, all computed in one forward pass thanks to the causal mask.

## Part 7. Adam, AdamW, and the learning-rate schedule

Adam keeps two running averages per weight, starting at 0, and corrects their bias toward 0:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\, g_t,\qquad s_t = \beta_2 s_{t-1} + (1-\beta_2)\, g_t^2,$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t},\qquad \hat s_t = \frac{s_t}{1-\beta_2^t},\qquad \theta_t = \theta_{t-1} - \gamma\, \frac{\hat m_t}{\sqrt{\hat s_t} + \epsilon}.$$

PyTorch's defaults: $\gamma = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

**Weight decay** $\lambda$ can enter in two places.
* *Adam with an L2 penalty* (`torch.optim.Adam(weight_decay=λ)`): add $\lambda\theta$ to the gradient, $g_t \leftarrow g_t + \lambda\theta_{t-1}$, before updating $m$ and $s$.
* *AdamW* (`torch.optim.AdamW`): decay as a separate step, outside $m$ and $s$: first $\theta \leftarrow \theta\,(1 - \gamma\lambda)$, then the Adam step with the plain gradient.

**Schedule.** Linear warmup over the first $W$ steps, then a cosine decay to 0 at step $N$:

$$\gamma(t) = \begin{cases} \gamma_{\text{peak}}\, t / W & t < W \\ \gamma_{\text{peak}} \cdot \tfrac12\left(1 + \cos\left(\pi\,\frac{t - W}{N - W}\right)\right) & t \ge W. \end{cases}$$

In [22]:
# TODO 7: one Adam / AdamW step, and the warmup + cosine schedule.
def adam_step(theta, g, m, s, t, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0, decoupled=False):
    """All arrays are numpy arrays of the same shape; t is the step number, starting at 1.
    decoupled=False: Adam with an L2 penalty (add weight_decay * theta to g).
    decoupled=True:  AdamW (multiply theta by 1 - lr * weight_decay, then the Adam step).
    Return (theta_new, m_new, s_new)."""
    raise NotImplementedError

def lr_at(t, peak=3e-4, W=500, N=10_000):
    """Linear warmup for t < W, then cosine decay from peak at t = W to 0 at t = N."""
    raise NotImplementedError

In [23]:
g1 = np.array([10.0, 0.1])
step = attempt(adam_step, np.zeros(2), g1, np.zeros(2), np.zeros(2), 1)
if step is not None:
    th1, m1, s1 = step
    print("                 weight 1    weight 2")
    for name, v in [("gradient g", g1), ("m1 = 0.1 g", m1), ("s1 = 0.001 g^2", s1), ("m_hat = m1/0.1", m1 / 0.1),
                    ("s_hat = s1/0.001", s1 / 0.001), ("Adam step", -th1), ("SGD step", 1e-3 * g1)]:
        print(f"{name:17s} {v[0]:10.5g}  {v[1]:10.5g}")
    assert np.allclose(m1, [1, 0.01]) and np.allclose(s1, [0.1, 1e-5])
    assert np.allclose(-th1, [0.001, 0.001], rtol=1e-6)
    th100 = adam_step(np.zeros(2), 100 * g1, np.zeros(2), np.zeros(2), 1)[0]
    print("every gradient x 100 -> Adam step", -th100, "(unchanged)")
    assert np.allclose(th100, th1)

    # several steps of Adam (+L2) and AdamW against torch.optim
    rng = np.random.default_rng(8)
    for decoupled, cls in [(False, torch.optim.Adam), (True, torch.optim.AdamW)]:
        theta0 = rng.normal(size=5); grads = rng.normal(size=(6, 5))
        w = torch.tensor(theta0, requires_grad=True)
        opt = cls([w], lr=0.01, weight_decay=0.1)
        th, m, s = theta0.copy(), np.zeros(5), np.zeros(5)
        for t, gt in enumerate(grads, 1):
            w.grad = torch.tensor(gt); opt.step()
            th, m, s = adam_step(th, gt, m, s, t, lr=0.01, weight_decay=0.1, decoupled=decoupled)
        assert np.allclose(th, w.detach().numpy(), atol=1e-10), cls.__name__
        print(f"6 steps match torch.optim.{cls.__name__}: ok")

    sched = [lr_at(t) for t in [0, 250, 500, 5250, 10_000]]
    print("lr at steps 0, 250, 500, 5250, 10000:", [f"{v:.2e}" for v in sched])
    assert np.allclose(sched, [0, 1.5e-4, 3e-4, 1.5e-4, 0], atol=1e-12)
    curve = np.array([lr_at(t) for t in range(10_001)])
    assert (np.diff(curve[:501]) > 0).all() and (np.diff(curve[500:]) <= 0).all()
    print("warmup rises to the peak at step 500, cosine falls to half the peak at step 5,250 and to 0 at 10,000: ok")

[TODO] adam_step is not implemented yet: skipping this check.


Both weights move by $\gamma = 0.001$ on the first step even though their gradients differ by 100x, and scaling every gradient by 100 changes nothing: Adam's learning rate is roughly *how far each weight moves per step*.

### SGD vs Adam on a ravine

The slides' loss is $\tfrac12(w_1^2 + 50\,w_2^2)$: steep in $w_2$, flat in $w_1$. Plain SGD multiplies $w_2$ by $(1 - 50\gamma)$ each step, so it diverges for $\gamma > 2/50 = 0.04$, and with $\gamma$ that small it crawls along $w_1$. We start both from $(8, 1.2)$, where the loss is 68, and run 40 steps with the best $\gamma$ from a grid.

In [24]:
if step is not None:
    loss_fn = lambda w: 0.5 * (w[0] ** 2 + 50 * w[1] ** 2)
    grad_fn = lambda w: np.array([w[0], 50 * w[1]])
    def run(method, lr, steps=40):
        w, m, s = np.array([8.0, 1.2]), np.zeros(2), np.zeros(2)
        for t in range(1, steps + 1):
            if method == "sgd":
                w = w - lr * grad_fn(w)
            else:
                w, m, s = adam_step(w, grad_fn(w), m, s, t, lr=lr)
        return loss_fn(w)
    print("start loss:", loss_fn(np.array([8.0, 1.2])))
    sgd_grid = np.round(np.arange(0.001, 0.0415, 0.001), 3)
    adam_grid = np.round(np.arange(0.01, 1.001, 0.01), 2)
    sgd = {lr: run("sgd", lr) for lr in sgd_grid}
    adam = {lr: run("adam", lr) for lr in adam_grid}
    best_sgd, best_adam = min(sgd, key=sgd.get), min(adam, key=adam.get)
    print(f"SGD : best gamma {best_sgd}, loss 68 -> {sgd[best_sgd]:.2f};  gamma 0.041 -> {sgd[0.041]:.0f} (diverges)")
    print(f"Adam: best gamma {best_adam}, loss 68 -> {adam[best_adam]:.4f}")
    print("Adam's final loss for nearby gammas:", ", ".join(f"{lr}: {adam[lr]:.4f}" for lr in [0.26, 0.27, 0.28, 0.59, 0.6, 0.61]))
    assert loss_fn(np.array([8.0, 1.2])) == 68 and abs(sgd[best_sgd] - 1.45) < 0.01 and sgd[0.041] > 68
    assert round(adam[best_adam], 3) == 0.003 and round(adam[0.27], 3) == 0.003 and adam[0.26] > 5 * adam[0.27]

This reproduces the slide: SGD's best is about 1.45, Adam's best about 0.003. Look at Adam's row of nearby learning rates, though: $\gamma = 0.27$ and $\gamma = 0.6$ both land near 0.003, while $\gamma = 0.26$ ends about ten times higher. After only 40 steps Adam is still oscillating around the minimum, so its "best" number is partly a lucky landing. Read the comparison as "orders of magnitude apart", not as a precise value.

### Why AdamW: where the decay lands

Give 500 weights large noisy gradients (scale 10) and 500 weights small ones (scale 0.1), with no real signal, and let weight decay ($\lambda = 0.1$) pull all of them toward 0 for 2,000 steps. With the L2 penalty, the decay term is divided by $\sqrt{\hat s}$ like everything else, so it is drowned by large gradients and amplified by small ones. AdamW applies the same decay rate to every weight.

In [25]:
if step is not None:
    def decay_demo(decoupled, n=500, steps=2000, lr=1e-2, lam=0.1):
        scale = np.r_[np.full(n, 10.0), np.full(n, 0.1)]
        th, m, s = np.ones(2 * n), np.zeros(2 * n), np.zeros(2 * n)
        rng = np.random.default_rng(1)
        for t in range(1, steps + 1):
            th, m, s = adam_step(th, scale * rng.normal(size=2 * n), m, s, t, lr=lr, weight_decay=lam, decoupled=decoupled)
        return np.abs(th[:n]).mean(), np.abs(th[n:]).mean()
    l2, adamw = decay_demo(False), decay_demo(True)
    print(f"mean |theta| after decay      large-gradient weights   small-gradient weights")
    print(f"Adam + L2 penalty                    {l2[0]:.3f}                    {l2[1]:.3f}")
    print(f"AdamW                                {adamw[0]:.3f}                    {adamw[1]:.3f}")
    assert l2[0] > 5 * l2[1] and 0.7 < adamw[0] / adamw[1] < 1.4

Under Adam with an L2 penalty the large-gradient weights barely decay while the small-gradient ones collapse toward 0; under AdamW both groups end up alike. That uneven decay is what "decoupled weight decay" removes, and why AdamW is the usual choice for Transformers.

## Summary

| Part | Idea | What you computed |
|---|---|---|
| 1 | Softmax, temperature, $1/\sqrt{D}$ | $\mathrm{var}(q\cdot k) = D$; the scaled vs raw softmax and the temperature table |
| 2 | Masks before the softmax | 15 masked entries in a $6\times6$ causal mask; padding columns; shapes for $B=32, T=42, D=64$; the leak test |
| 3 | Multi-head attention | matches `nn.MultiheadAttention`; permutation equivariance without positions; the KV cache and its 2.1 GB |
| 4 | Parameter counts | 11,620 for the one-layer model; $12D^2 + 13D$ per pre-LN block; GPT-2 small 124M, XL 1.56B, GPT-3 175B |
| 5 | BPE | `l o w e r → lo w e r → low e r → low er`; unseen words split into known pieces |
| 6 | Next-token loss | shift by one, ignore pads; perplexity $V$ for a uniform guess and $2^{2/7}$ for a bigram model |
| 7 | Adam, AdamW, schedule | the first Adam step; agreement with `torch.optim`; SGD 1.45 vs Adam 0.003; even decay with AdamW; half the peak at step 5,250 |

**Credits.** Self-attention, multi-head attention, $1/\sqrt{D}$ scaling and learning-rate warmup: A. Vaswani et al., "Attention Is All You Need", NeurIPS 2017; the step-by-step $Q/K/V$ pictures the notation follows are from Prof. Hung-yi Lee's ML 2021 lectures (NTU). Layer normalization: J. L. Ba, J. R. Kiros & G. E. Hinton, arXiv 2016; the pre-LN placement: A. Radford et al. (GPT-2), 2019, analysed by R. Xiong et al., ICML 2020. GPT model sizes: A. Radford et al., 2018 and 2019; T. Brown et al. (GPT-3), NeurIPS 2020. Byte-pair encoding for subwords: R. Sennrich, B. Haddow & A. Birch, ACL 2016, after P. Gage's compression algorithm (1994). Temperature sampling and degeneration: A. Holtzman et al., ICLR 2020. Adam: D. P. Kingma & J. Ba, ICLR 2015; AdamW: I. Loshchilov & F. Hutter, ICLR 2019; cosine schedule: I. Loshchilov & F. Hutter (SGDR), ICLR 2017. The weight-tied GPT layout follows A. Karpathy's nanoGPT. The toy corpus, the ravine start point and all random data are made up for teaching.

In [26]:
print(f"Total notebook runtime: {time.time() - T_START:.0f} s")

Total notebook runtime: 0 s
